# De Qiskit a Polypus

Guía de referencia autocontenida para quien ya sabe construir y ejecutar circuitos con Qiskit y solo necesita el mapeo directo a Polypus, sin pasar por una introducción desde cero.

## Construir un circuito

Mismo patrón, misma idea de registro de qubits, con encadenado de métodos en Polypus:

In [ ]:
# Qiskit
from qiskit import QuantumCircuit

qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)
qc_qiskit.cx(0, 1)
qc_qiskit.measure_all()

# Polypus
import polypus

qc_polypus = polypus.Circuit(2).h(0).cx(0, 1).measure_all()

## Puertas: mismos nombres, cuidado con el orden de los argumentos

Las puertas sin ángulo se llaman exactamente igual, con los mismos argumentos en el mismo orden:

| Puerta | Qiskit y Polypus |
|---|---|
| Hadamard | `qc.h(q)` |
| Pauli X / Y / Z | `qc.x(q)`, `qc.y(q)`, `qc.z(q)` |
| S / Sdg | `qc.s(q)`, `qc.sdg(q)` |
| T / Tdg | `qc.t(q)`, `qc.tdg(q)` |
| CNOT | `qc.cx(control, target)` |
| CZ | `qc.cz(control, target)` |
| SWAP | `qc.swap(q0, q1)` |
| Medida | `qc.measure(q, c)`, `qc.measure_all()` |

Las puertas con ángulo tienen el mismo nombre, pero **el orden de los argumentos está invertido**: Qiskit pone el ángulo primero, Polypus lo pone al final.

| Puerta | Qiskit | Polypus |
|---|---|---|
| Rotación X / Y / Z | `qc.rx(theta, q)` | `qc.rx(q, theta)` |
| Fase controlada | `qc.cp(theta, control, target)` | `qc.cp(control, target, theta)` |
| ZZ / XX | `qc.rzz(theta, q0, q1)` | `qc.rzz(q0, q1, theta)` |
| U genérica | `qc.u(theta, phi, lam, q)` | `qc.u(q, theta, phi, lam)` |

Copiar y pegar una llamada de Qiskit con ángulo tal cual falla alto y claro, no en silencio: Polypus espera un entero en la posición del qubit, así que un ángulo ahí da un `TypeError` inmediato, no un resultado incorrecto sin avisar.

In [ ]:
import math

try:
    polypus.Circuit(1).rx(math.pi / 3, 0)  # orden de Qiskit, por error
except TypeError as e:
    print(e)

## Ejecutar

Qiskit ya no tiene `execute()`, desde la versión 1.0 se ejecuta contra un backend directamente. El `AerSimulator` de Qiskit corresponde al `infrastructure="local"` de Polypus:

In [ ]:
# Qiskit
from qiskit_aer import AerSimulator

backend = AerSimulator()
counts_qiskit = backend.run(qc_qiskit, shots=1000, seed_simulator=42).result().get_counts()

# Polypus
result = polypus.run_quantum_circuit(
    qc_polypus, shots=1000, infrastructure="local", seed=42
)
counts_polypus = result.counts[0]  # counts[0]: una ejecución, ver 03a

print(counts_qiskit)
print(counts_polypus)

Los dos resultados salen idénticos con la misma semilla: el backend `"aer"` de Polypus, el que usa `infrastructure="local"` por defecto, es el mismo `AerSimulator` de Qiskit por debajo. La diferencia real está en la forma del resultado: `get_counts()` de Qiskit devuelve el diccionario directamente, `result.counts` de Polypus lo envuelve en una lista, con un elemento por ejecución en paralelo solicitada (`n_qpus`); con una sola ejecución, como aquí, es una lista de un elemento.

## Sin reescribir nada: Polypus ejecuta un `QuantumCircuit` de Qiskit tal cual

`run_quantum_circuit` acepta un `QuantumCircuit` de Qiskit directamente, sin convertirlo antes a `polypus.Circuit`. Un circuito de Qiskit ya escrito no necesita tocarse para ganar acceso al resto de infraestructuras de Polypus:

In [ ]:
result = polypus.run_quantum_circuit(qc_qiskit, shots=1000, infrastructure="local")
print(result.counts[0])

## Escalar más allá de un simulador local

Donde Qiskit usa un backend, un provider o una sesión distinta según el proveedor de hardware, Polypus cambia un único argumento, `infrastructure`: `"local"`, visto arriba, `"cunqa"` para QPUs simuladas repartidas por SLURM, o `"qmio"` para hardware cuántico real. El mismo circuito, sin cambios, se mueve entre las tres.

## Entrenar circuitos variacionales

Qiskit no trae un optimizador propio para esto: lo habitual es un bucle manual con `scipy.optimize`, bindeando parámetros y llamando al backend en cada evaluación:

In [ ]:
# Qiskit + scipy
from qiskit.circuit import Parameter
from scipy.optimize import differential_evolution

theta = Parameter("theta")
qc_entrenar = QuantumCircuit(1)
qc_entrenar.h(0)
qc_entrenar.rz(theta, 0)
qc_entrenar.h(0)
qc_entrenar.measure_all()


def coste(params):
    qc_bound = qc_entrenar.assign_parameters({theta: params[0]})
    counts = backend.run(qc_bound, shots=500).result().get_counts()
    total = sum(counts.values())
    p0 = counts.get("0", 0) / total
    return -p0  # differential_evolution siempre minimiza


resultado_scipy = differential_evolution(
    coste, bounds=[(0, 2 * math.pi)], seed=3, maxiter=20, popsize=10
)
print(resultado_scipy.x, -resultado_scipy.fun)

`polypus.train()` sustituye todo ese bucle manual, sin necesidad de reescribir el circuito: acepta el mismo `QuantumCircuit` con `Parameter` de Qiskit de arriba tal cual, se le pasa una función que recibe un bitstring en vez de un diccionario de cuentas completo, y Polypus se encarga de bindear, ejecutar y promediar por dentro:

In [ ]:
# Polypus
def coste_bitstring(bitstring):
    return 1.0 if bitstring == "0" else 0.0


resultado_polypus = polypus.train(
    qc_entrenar,
    polypus.DE(generations=20, population_size=10, seed=3),
    shots=500,
    n_qpus=1,
    dimensions=1,
    expectation_function=coste_bitstring,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="qiskit-a-polypus",
)
print(resultado_polypus.best_params, resultado_polypus.best_fitness)

Los dos ángulos encontrados no coinciden como número, pero son equivalentes por periodicidad: los dos dan `P(0) = 1.0`. Un detalle de signo al migrar: `differential_evolution` de scipy siempre minimiza, de ahí el signo negativo en `coste`; `DE` de Polypus siempre maximiza, así que `coste_bitstring` no necesita invertir nada. Los otros optimizadores de Polypus, `PSO` y `QNG`, comparten la misma llamada a `train()`, cambiando solo el objeto pasado como `method`.

## Resumen

La mayoría del código de Qiskit ya construido se ejecuta contra Polypus sin reescribirlo: mismos nombres de puerta, con el orden de argumentos invertido en las que llevan ángulo, y un `QuantumCircuit` completo se acepta tal cual en `run_quantum_circuit` y en `train()`. Lo que cambia de verdad es la forma de escalar, `infrastructure` en vez de backends y providers, y el entrenamiento, `train()` en vez de un bucle manual con `scipy`.